# EfficientNet-B0 Training (no Optuna)

Simple path: train EfficientNet-B0 on FER → save `best_model_efficientnet_b0.h5`.

Uses fixed params from your last good Optuna run (no search, less RAM, faster to a downloadable model).

## Where to run
- **Kaggle (recommended):** dataset `fahadullaha/facial-emotion-recognition-dataset`, GPU on, Run All
- **Colab:** set `PLATFORM = "colab"`

## After download
```
smartshop/mood_model/artifacts/models/best_model_efficientnet_b0.h5
```
Then Egyptian fine-tune: `colab_finetune_efficientnet_egypt.ipynb`


In [ ]:
# 0) Config
PLATFORM = "kaggle"   # "kaggle" | "colab"

EPOCHS_HEAD = 10
EPOCHS_FT = 12
IMAGE_SIZE = 96
SEED = 123
BATCH_SIZE = 16  # from first Optuna best trial

# Fixed params from your FIRST Optuna run — Trial 1 (best val_loss ~1.52)
# Better than the later 5-trial best (~1.70). No need to re-run Optuna.
PARAMS = {
    "dropout1": 0.31941327659912944,
    "dense_units": 128,
    "dropout2": 0.3594654121525515,
    "head_lr": 0.0010208192281310883,
    "ft_lr": 1.2927728217910711e-05,
    "n_last": 20,
}

print("PLATFORM", PLATFORM)
print("BATCH_SIZE", BATCH_SIZE)
print("PARAMS", PARAMS)


In [ ]:
# 1) Imports
import os
import json
import random
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("TF", tf.__version__)
print("GPU", tf.config.list_physical_devices("GPU"))

if PLATFORM == "kaggle":
    WORK = Path("/kaggle/working")
else:
    WORK = Path("/content/effnet_run")
    WORK.mkdir(parents=True, exist_ok=True)

OUT_DIR = WORK / "efficientnet_b0_baseline"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR", OUT_DIR)


In [ ]:
# 2) Dataset path
if PLATFORM == "kaggle":
    DATASET_DIR = Path("/kaggle/input/facial-emotion-recognition-dataset/processed_data")
    if not DATASET_DIR.exists():
        hits = list(Path("/kaggle/input").glob("**/processed_data"))
        if not hits:
            raise FileNotFoundError("Add Kaggle dataset: fahadullaha/facial-emotion-recognition-dataset")
        DATASET_DIR = hits[0]
else:
    !pip -q install kagglehub
    import kagglehub
    root = Path(kagglehub.dataset_download("fahadullaha/facial-emotion-recognition-dataset"))
    hits = list(root.glob("**/processed_data"))
    DATASET_DIR = hits[0] if hits else root

print("DATASET_DIR", DATASET_DIR)


In [ ]:
# 3) Load dataset + class names
probe = tf.keras.preprocessing.image_dataset_from_directory(
    str(DATASET_DIR),
    seed=SEED,
    shuffle=False,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=32,
)
CLASS_NAMES = probe.class_names
N_CLASSES = len(CLASS_NAMES)
print("Classes:", CLASS_NAMES)

labels_all = []
for _, y in probe:
    labels_all.extend(y.numpy().tolist())
labels_all = np.array(labels_all)
print("Samples:", len(labels_all))
print({CLASS_NAMES[i]: int((labels_all == i).sum()) for i in range(N_CLASSES)})

full = tf.keras.preprocessing.image_dataset_from_directory(
    str(DATASET_DIR),
    seed=SEED,
    shuffle=True,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=None,
)


In [ ]:
# 4) Train/val/test split + pipelines
AUTOTUNE = tf.data.AUTOTUNE
n = len(labels_all)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
n_test = n - n_train - n_val

full = full.shuffle(n, seed=SEED, reshuffle_each_iteration=False)
train_raw = full.take(n_train)
val_raw = full.skip(n_train).take(n_val)
test_raw = full.skip(n_train + n_val)

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.15),
], name="aug")

def prep(ds, training, batch_size):
    def _map(x, y):
        x = tf.cast(x, tf.float32)
        if training:
            x = data_augmentation(x)
        return x, y
    ds = ds.map(_map, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(1024, seed=SEED)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

# class weights from train labels
y_train_list = [int(y.numpy()) for _, y in train_raw]
y_train = np.array(y_train_list)
cw = compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=y_train)
CLASS_WEIGHT = {int(i): float(w) for i, w in enumerate(cw)}
print("Class weights:", CLASS_WEIGHT)
print("Split sizes:", n_train, n_val, n_test)

# recreate streams (iteration consumed train_raw)
full2 = tf.keras.preprocessing.image_dataset_from_directory(
    str(DATASET_DIR),
    seed=SEED,
    shuffle=True,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=None,
).shuffle(n, seed=SEED, reshuffle_each_iteration=False)
train_raw = full2.take(n_train)
val_raw = full2.skip(n_train).take(n_val)
test_raw = full2.skip(n_train + n_val)

train_ds = prep(train_raw, True, BATCH_SIZE)
val_ds = prep(val_raw, False, BATCH_SIZE)
test_ds = prep(test_raw, False, BATCH_SIZE)
print("Pipelines ready")


In [ ]:
# 5) Build EfficientNet-B0 ONLY (never overwrite with CNN)

def build_efficientnet_b0(
    n_classes=N_CLASSES,
    dropout1=0.3,
    dense_units=256,
    dropout2=0.3,
    head_lr=1e-3,
):
    base = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    )
    base.trainable = False

    inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout1)(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout2)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs, name="efficientnet_b0_emotion")

    model.compile(
        optimizer=optimizers.Adam(learning_rate=head_lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model, base


def unfreeze_last_layers(base, n_last=30):
    base.trainable = True
    for layer in base.layers[:-n_last]:
        layer.trainable = False
    for layer in base.layers[-n_last:]:
        layer.trainable = True


def evaluate_ds(model, ds, split_name="set"):
    y_true, y_pred = [], []
    for images, labels in ds:
        probs = model.predict(images, verbose=0)
        y_true.extend(labels.numpy().tolist())
        y_pred.extend(np.argmax(probs, axis=1).tolist())
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "y_true": y_true,
        "y_pred": y_pred,
    }
    print(
        f"{split_name}: accuracy={out['accuracy']:.4f}  "
        f"macro_f1={out['macro_f1']:.4f}  weighted_f1={out['weighted_f1']:.4f}"
    )
    return out


def has_efficientnet(m):
    if "efficientnet" in m.name.lower():
        return True
    for l in m.layers:
        if "efficientnet" in l.name.lower():
            return True
        if hasattr(l, "layers") and any("efficientnet" in sl.name.lower() for sl in l.layers):
            return True
    return False

model, base = build_efficientnet_b0(
    dropout1=float(PARAMS["dropout1"]),
    dense_units=int(PARAMS["dense_units"]),
    dropout2=float(PARAMS["dropout2"]),
    head_lr=float(PARAMS["head_lr"]),
)
assert has_efficientnet(model), "Builder did not create EfficientNet"
model.summary()
print("EfficientNet check: OK")


In [ ]:
# 6) Stage 1 — train head (EfficientNet frozen)
ckpt_path = str(OUT_DIR / "best_model_efficientnet_b0.keras")
callbacks_head = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2),
    ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True),
]

print("Stage 1: train head")
hist1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    class_weight=CLASS_WEIGHT,
    callbacks=callbacks_head,
    verbose=1,
)

print("After Stage 1 (VAL):")
_ = evaluate_ds(model, val_ds, "VAL")


In [ ]:
# 7) Stage 2 — fine-tune last N EfficientNet layers
unfreeze_last_layers(base, int(PARAMS["n_last"]))
model.compile(
    optimizer=optimizers.Adam(float(PARAMS["ft_lr"])),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks_ft = [
    EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2),
    ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True),
]

print("Stage 2: fine-tune last", PARAMS["n_last"], "layers")
hist2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FT,
    class_weight=CLASS_WEIGHT,
    callbacks=callbacks_ft,
    verbose=1,
)

print("After Stage 2 (VAL):")
_ = evaluate_ds(model, val_ds, "VAL")


In [ ]:
# 8) Evaluate + save best_model_efficientnet_b0.h5
print("Final TEST metrics:")
metrics = evaluate_ds(model, test_ds, "TEST")
print(classification_report(metrics["y_true"], metrics["y_pred"], target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(metrics["y_true"], metrics["y_pred"])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("EfficientNet-B0 — TEST")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix_efficientnet_b0.png", dpi=150)
plt.show()

assert has_efficientnet(model), "Model is not EfficientNet — abort save"
print("EfficientNet check: OK")

h5_path = OUT_DIR / "best_model_efficientnet_b0.h5"
easy = WORK / "best_model_efficientnet_b0.h5"
model.save(str(h5_path))
model.save(str(easy))

meta = {
    "backbone": "EfficientNetB0",
    "image_size": IMAGE_SIZE,
    "class_names": CLASS_NAMES,
    "params": PARAMS,
    "batch_size": BATCH_SIZE,
    "test_accuracy": metrics["accuracy"],
    "test_macro_f1": metrics["macro_f1"],
    "test_weighted_f1": metrics["weighted_f1"],
    "note": "EfficientNet-B0 baseline (no Optuna) for Egyptian fine-tuning",
}
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved:", easy)
print("Also:", h5_path)
print(json.dumps(meta, indent=2))


In [ ]:
# 9) Download
if PLATFORM == "colab":
    from google.colab import files
    files.download(str(WORK / "best_model_efficientnet_b0.h5"))
else:
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(WORK / "best_model_efficientnet_b0.h5")))
        display(FileLink(str(OUT_DIR / "metrics.json")))
    except Exception as e:
        print("Download from Kaggle Output:", WORK / "best_model_efficientnet_b0.h5")
        print(e)

print("\nPut on Mac:")
print("smartshop/mood_model/artifacts/models/best_model_efficientnet_b0.h5")
print("\nNext: colab_finetune_efficientnet_egypt.ipynb")


## Done when
1. `metrics.json` says `"backbone": "EfficientNetB0"`
2. You downloaded `best_model_efficientnet_b0.h5`
3. You have TEST **accuracy**, **macro-F1**, **weighted-F1**, and the classification report
